# 🤖 03 — Model Training & Evaluation
Train and evaluate VADER + DistilBERT models.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from data.data_loader import load_sample_data
from preprocessing.cleaner import TextCleaner
from models.sentiment_vader import VADERSentimentAnalyzer

df = load_sample_data()
print(f'Dataset loaded: {len(df)} reviews')
df.head()

## 1. Text Preprocessing

In [ ]:
cleaner = TextCleaner(remove_stopwords=True, lemmatize=False)

df['clean_text'] = cleaner.clean_series(df['text'])
df['tokens']     = cleaner.tokenize_series(df['text'])

print('Before cleaning:')
print(df['text'].iloc[0])
print('\nAfter cleaning:')
print(df['clean_text'].iloc[0])
print('\nTokens:')
print(df['tokens'].iloc[0])

## 2. VADER Sentiment Analysis

In [ ]:
vader = VADERSentimentAnalyzer()
df = vader.predict_batch(df, text_col='text')

distribution = vader.get_sentiment_distribution(df)
print('Sentiment Distribution:')
for label, info in distribution.items():
    print(f'  {label}: {info["count"]} reviews ({info["percentage"]}%)')

df[['text', 'vader_label', 'vader_compound']].head(10)

In [ ]:
# Visualize VADER results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label distribution
label_counts = df['vader_label'].value_counts()
colors = {'Positive': '#22c55e', 'Negative': '#ef4444', 'Neutral': '#94a3b8'}
bar_colors = [colors.get(l, '#6366f1') for l in label_counts.index]

axes[0].bar(label_counts.index, label_counts.values, color=bar_colors)
axes[0].set_title('VADER Sentiment Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

# Compound score distribution
axes[1].hist(df['vader_compound'], bins=20, color='#6366f1', edgecolor='white')
axes[1].axvline(x=0.05, color='#22c55e', linestyle='--', label='Positive threshold')
axes[1].axvline(x=-0.05, color='#ef4444', linestyle='--', label='Negative threshold')
axes[1].set_title('Compound Score Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('../assets/vader_results.png', dpi=150)
plt.show()

## 3. VADER vs Rating Correlation (Gold Standard)

In [ ]:
# Create pseudo ground truth from star ratings
def rating_to_sentiment(rating):
    if rating >= 4: return 'Positive'
    elif rating <= 2: return 'Negative'
    else: return 'Neutral'

df['true_sentiment'] = df['rating'].apply(rating_to_sentiment)

print('Classification Report (VADER vs Star Rating):')
print(classification_report(df['true_sentiment'], df['vader_label']))

# Confusion matrix
labels = ['Positive', 'Negative', 'Neutral']
cm = confusion_matrix(df['true_sentiment'], df['vader_label'], labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix — VADER')
plt.ylabel('True (Rating-based)')
plt.xlabel('Predicted (VADER)')
plt.tight_layout()
plt.savefig('../assets/confusion_matrix.png', dpi=150)
plt.show()

## 4. DistilBERT (Optional — GPU recommended)

In [ ]:
# Uncomment to run DistilBERT (requires ~500MB download)
# from models.sentiment_bert import BERTSentimentAnalyzer
# 
# bert = BERTSentimentAnalyzer(use_gpu=False)
# df = bert.predict_batch(df, text_col='text', batch_size=16)
# 
# print('BERT Results:')
# print(df['bert_label'].value_counts())
# 
# # Model agreement
# agreement = (df['vader_label'] == df['bert_label']).mean()
# print(f'\nModel Agreement: {agreement:.1%}')

print('DistilBERT block is commented out.')
print('Uncomment and run when ready.')

In [ ]:
# Save results
df.to_csv('../data/analyzed_reviews.csv', index=False)
print('Results saved to data/analyzed_reviews.csv')